In [12]:
import tensorflow as tf
print("Num GPUs:", len(tf.config.list_physical_devices('GPU')))

Num GPUs: 1


In [13]:
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [14]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from joblib import dump

In [15]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = r"C:\Users\noeld\Downloads\Sentiment Analysis NLP\data\Twitter data final.csv"

In [ ]:
# model = r"C:\Users\noeld\Downloads\Sentiment Analysis NLP\sentiment_model.h5"   # trained Keras model
# tokenizer = r"C:\Users\noeld\Downloads\Sentiment Analysis NLP\tokenizer.joblib"  # saved tokenizer 

# if LSTM or BiLSTM is used as final model

In [18]:
df = pd.read_csv(data)

In [19]:
df.shape

(75680, 4)

In [20]:
df.head()

,Tweet ID,Entity,Sentiment,Tweet content
0,1,Amazon,Negative,<unk> wtf.
1,1,Amazon,Negative,@amazon wtf?
2,1,Amazon,Negative,@ amazon wtf.
3,1,Amazon,Negative,@ amazon wtf.
4,1,Amazon,Negative,@amazon wtf .


In [21]:
# Shuffle the dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [22]:
df.head()

,Tweet ID,Entity,Sentiment,Tweet content
0,7175,johnson&johnson,Negative,We don't trust Johnson & Johnson tho. They lie...
1,3407,Facebook,Negative,"Comment "" one group I know re @Facebook and it..."
2,7152,johnson&johnson,Negative,Create a problem for which you already have a ...
3,2461,Borderlands,Positive,"I feel like I've spilled something here, when ..."
4,5145,GrandTheftAuto(GTA),Neutral,I don't mind waiting for GTA 6 if it doesn't m...


In [23]:
df = df[["Tweet content", "Sentiment"]].copy()

In [24]:
df.head()

,Tweet content,Sentiment
0,We don't trust Johnson & Johnson tho. They lie...,Negative
1,"Comment "" one group I know re @Facebook and it...",Negative
2,Create a problem for which you already have a ...,Negative
3,"I feel like I've spilled something here, when ...",Positive
4,I don't mind waiting for GTA 6 if it doesn't m...,Neutral


In [25]:
# Reproducibility
RANDOM_STATE = 42

# Model & text hyperparameters
MAX_NUM_WORDS = 20000         # vocabulary size for Tokenizer (It will do a frquency count and keep the top MAX_NUM_WORDS)

MAX_SEQUENCE_LENGTH = 40      # max tokens per tweet (for padding)

EMBEDDING_DIM = 100           # dimension of embedding vectors

EPOCHS = 100                  # max epochs (EarlyStopping will likely stop earlier)
BATCH_SIZE = 32               # batch size

In this project, fixed hyperparameters are used to ensure reproducibility and stable model performance. A random seed `RANDOM_STATE = 42` is set so that the model produces the same results every time it is trained.

For text processing, the vocabulary size is limited to the top 20,000 most frequent words, which helps reduce noise and memory usage. Each tweet is converted into a sequence of 40 tokens, ensuring uniform input length through padding.

An embedding dimension of 100 is chosen to represent words in a meaningful numerical form while keeping the model computationally efficient. The model is trained with a batch size of 32 to balance learning stability and speed. Although the maximum number of epochs is set to 100, Early Stopping is used so the training stops automatically when performance no longer improves, preventing overfitting.

Overall, these settings help the model learn efficiently while maintaining consistency and generalization.

In [26]:
df["Sentiment"].value_counts( dropna = False)

,count
Sentiment,
Negative,22808
Positive,21108
Neutral,18603
Irrelevant,13161


In [27]:
df = df[df["Sentiment"] != "Irrelevant"]

In [28]:
df["Sentiment"].value_counts()

,count
Sentiment,
Negative,22808
Positive,21108
Neutral,18603


In [29]:
df["Tweet content"].value_counts( dropna = False)

,count
Tweet content,
NaN,571
It is not the first time that the EU Commission has taken such a step.,139
"At the same time, despite the fact that there are currently some 100 million people living below the poverty line, most of them do not have access to health services and do not have access to health care, while most of them do not have access to health care.",139
,139
<unk>,109
...,...
Fuck this game,1
I’m liking the new update for,1
Is Israel trolling us,1


In [30]:
df = df.dropna(subset=["Tweet content"])

In [31]:
df = df[df["Tweet content"].str.strip() != ""]

In [32]:
df = df[df["Tweet content"].str.lower() != "<unk>"]

In [33]:
print(df["Tweet content"].isna().sum())

0


In [34]:
print(df.shape)

(61700, 2)


In [35]:
# Mapping between sentiment strings and numeric IDs
LABEL_TO_ID = {"Negative": 0, "Neutral": 1, "Positive": 2}

In [36]:
LABEL_TO_ID

{'Negative': 0, 'Neutral': 1, 'Positive': 2}

In [37]:
k,v = [('Negative', 0), ('Neutral', 1), ('Positive', 2)][0]

In [38]:
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}
ID_TO_LABEL

{0: 'Negative', 1: 'Neutral', 2: 'Positive'}

In [39]:
NUM_CLASSES = len(LABEL_TO_ID)

In [40]:
NUM_CLASSES

3

### Load the Dataset and Overview

In this step we:

- Load `Twitter_Data.csv`
- Inspect the first few rows
- Check the distribution of the `sentiment` labels
- Look at basic statistics of tweet length

This gives us an understanding of class balance and the nature of the text.

In [41]:
# Load raw dataset and basic EDA

print("Raw shape:", df.shape)

Raw shape: (61700, 2)


In [42]:
print("Columns:", df.columns.tolist())

Columns: ['Tweet content', 'Sentiment']


In [43]:
display(df.head())

,Tweet content,Sentiment
0,We don't trust Johnson & Johnson tho. They lie...,Negative
1,"Comment "" one group I know re @Facebook and it...",Negative
2,Create a problem for which you already have a ...,Negative
3,"I feel like I've spilled something here, when ...",Positive
4,I don't mind waiting for GTA 6 if it doesn't m...,Neutral


In [44]:
# Sentiment label distribution

df["Sentiment"].value_counts(dropna=False)

,count
Sentiment,
Negative,22547
Positive,20849
Neutral,18304


In [45]:
# Simple length distribution of raw text
df["Text length"] = df["Tweet content"].astype(str).str.len()

In [46]:
df.head()

,Tweet content,Sentiment,Text length
0,We don't trust Johnson & Johnson tho. They lie...,Negative,81
1,"Comment "" one group I know re @Facebook and it...",Negative,70
2,Create a problem for which you already have a ...,Negative,85
3,"I feel like I've spilled something here, when ...",Positive,88
4,I don't mind waiting for GTA 6 if it doesn't m...,Neutral,174


In [47]:
df["Text length"].describe()

,Text length
count,61700.000000
mean,109.057439
std,79.542792
min,1.000000
25%,47.000000
50%,91.000000
75%,153.000000
max,957.000000


### NLP preprocessing – cleaning text

We now:

- Define a `clean_text()` function that:
  - Removes URLs  
  - Removes @mentions  
  - Keeps **letters, spaces, `!`, and `?`** (sentiment-rich punctuation)  
  - Lowercases text  
  - Collapses multiple spaces

- Drop rows with missing `text` or `sentiment`
- Apply `clean_text()` to create a `clean_text` column
- Drop very short cleaned tweets
- Map sentiment labels to numeric IDs (0 = negative, 1 = neutral, 2 = positive)

We will **reuse `clean_text()` in the deployment app** so training and inference preprocessing match.

In [ ]:
# Define cleaning function and apply it

def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+", " ", text)            # remove URLs
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)     # remove @mentions

    # Keep letters, spaces, and basic sentiment punctuation ! and ?
    text = re.sub(r"[^a-zA-Z\s!?]", " ", text)

    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [49]:
# Drop rows with missing text or sentiment
df = df.dropna(subset=["Tweet content", "Sentiment"]).copy()

In [50]:
# Apply cleaning
df["Clean tweet"] = df["Tweet content"].apply(clean_text)

In [51]:
df.head()

,Tweet content,Sentiment,Text length,Clean tweet
0,We don't trust Johnson & Johnson tho. They lie...,Negative,81,we don t trust johnson johnson tho they lied a...
1,"Comment "" one group I know re @Facebook and it...",Negative,70,comment one group i know re and its fundraisin...
2,Create a problem for which you already have a ...,Negative,85,create a problem for which you already have a ...
3,"I feel like I've spilled something here, when ...",Positive,88,i feel like i ve spilled something here when i...
4,I don't mind waiting for GTA 6 if it doesn't m...,Neutral,174,i don t mind waiting for gta if it doesn t mea...


In [52]:
# Drop very short cleaned tweets (1–2 characters)
df = df[df["Clean tweet"].str.len() > 2]

In [53]:
# Map labels to IDs
df["Label id"] = df["Sentiment"].map(LABEL_TO_ID)

In [54]:
df.head()

,Tweet content,Sentiment,Text length,Clean tweet,Label id
0,We don't trust Johnson & Johnson tho. They lie...,Negative,81,we don t trust johnson johnson tho they lied a...,0
1,"Comment "" one group I know re @Facebook and it...",Negative,70,comment one group i know re and its fundraisin...,0
2,Create a problem for which you already have a ...,Negative,85,create a problem for which you already have a ...,0
3,"I feel like I've spilled something here, when ...",Positive,88,i feel like i ve spilled something here when i...,2
4,I don't mind waiting for GTA 6 if it doesn't m...,Neutral,174,i don t mind waiting for gta if it doesn t mea...,1


In [55]:
df["Label id"].isnull().sum()

np.int64(0)

In [56]:
print("After cleaning, shape:", df.shape)

After cleaning, shape: (60979, 5)


In [57]:
display(df[["Tweet content", "Clean tweet", "Sentiment", "Label id"]].head())

,Tweet content,Clean tweet,Sentiment,Label id
0,We don't trust Johnson & Johnson tho. They lie...,we don t trust johnson johnson tho they lied a...,Negative,0
1,"Comment "" one group I know re @Facebook and it...",comment one group i know re and its fundraisin...,Negative,0
2,Create a problem for which you already have a ...,create a problem for which you already have a ...,Negative,0
3,"I feel like I've spilled something here, when ...",i feel like i ve spilled something here when i...,Positive,2
4,I don't mind waiting for GTA 6 if it doesn't m...,i don t mind waiting for gta if it doesn t mea...,Neutral,1


### Split into train and test sets

We now split:

- `X` – the features (cleaned tweets)
- `y` – the numeric labels (0/1/2)

We use a **stratified train–test split** so that the class distribution is similar
in the train and test sets.

In [ ]:
# Train–test split (stratified)

X = df["Clean tweet"].values   # numpy array of strings
y = df["Label id"].values      # numpy array of ints

In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

In [60]:
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 48783, Test size: 12196


In [61]:
print("Train label distribution:")
print(pd.Series(y_train).map(ID_TO_LABEL).value_counts())

Train label distribution:
Negative    17851
Positive    16490
Neutral     14442
Name: count, dtype: int64


In [62]:
print("Test label distribution:")
print(pd.Series(y_test).map(ID_TO_LABEL).value_counts())

Test label distribution:
Negative    4463
Positive    4122
Neutral     3611
Name: count, dtype: int64


### Tokenization and sequence padding

Steps:

1. Create a Keras `Tokenizer` with a fixed vocabulary size and `<OOV>` token  
2. Fit it **only on training texts** (best practice)  
3. Convert train and test texts to integer sequences  
4. Pad/truncate sequences to a fixed length (`MAX_SEQUENCE_LENGTH`)

Outputs:

- `X_train_pad`: (n_train, max_len) integer array  
- `X_test_pad`: (n_test, max_len) integer array

Tokenization & padding

In [63]:
# Tokenizer and padded sequences

tokenizer = Tokenizer(num_words=MAX_NUM_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)   # fit only on training data

In [64]:
# Convert text to sequences of word IDs
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

In [65]:
# Pad / truncate to fixed length
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

print("X_train_pad shape:", X_train_pad.shape)
print("X_test_pad shape :", X_test_pad.shape)

X_train_pad shape: (48783, 40)
X_test_pad shape : (12196, 40)


### LSTM model

In [ ]:
# Build LSTM model
from tensorflow.keras.layers import LSTM

lstm_model = models.Sequential([
    layers.Embedding(input_dim=MAX_NUM_WORDS,
    output_dim=EMBEDDING_DIM,
    input_length=MAX_SEQUENCE_LENGTH
    ),
    
    layers.SpatialDropout1D(0.1),
    
    LSTM(128),
    
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    
    layers.Dense(NUM_CLASSES, activation="softmax")
    ])

lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [67]:
lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

#### Train the model with EarlyStopping and class weights

In [68]:
# ---------------------------------------------------------
# Compute class weights and train LSTM model
# ---------------------------------------------------------

# Ensure numeric arrays
X_train_pad = np.asarray(X_train_pad, dtype="int32")
X_test_pad  = np.asarray(X_test_pad, dtype="int32")
y_train     = np.asarray(y_train, dtype="int32")
y_test      = np.asarray(y_test, dtype="int32")

In [69]:
# Compute class weights to handle any imbalance / ignored classes
classes = np.array([0, 1, 2])  # Negative, Neutral, Positive

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

In [70]:
class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_array)}

In [71]:
print("Class weights (LSTM):", class_weight_dict)

Class weights (LSTM): {0: 0.9109293596997368, 1: 1.1259520841988644, 2: 0.9861127956337175}


In [72]:
# EarlyStopping callback
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# Train LSTM model
history_lstm = lstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_test_pad, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1,
    class_weight=class_weight_dict
)

Epoch 1/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - accuracy: 0.4443 - loss: 1.0195 - val_accuracy: 0.6534 - val_loss: 0.7386
Epoch 2/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.7744 - loss: 0.5744 - val_accuracy: 0.8537 - val_loss: 0.4043
Epoch 3/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9027 - loss: 0.2700 - val_accuracy: 0.8767 - val_loss: 0.3587
Epoch 4/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9313 - loss: 0.1870 - val_accuracy: 0.8891 - val_loss: 0.3203
Epoch 5/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9466 - loss: 0.1457 - val_accuracy: 0.8963 - val_loss: 0.3143
Epoch 6/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.9553 - loss: 0.1164 - val_accuracy: 0.9048 - val_loss: 0.3110
Epoch 7/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 18s 9ms/step - accuracy: 0.9618 - loss: 0.0968 - val_accuracy: 0.9071 - val_loss: 0.3387
Epoch 8/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9679 

#### Evaluate the model on the test set

In [ ]:
y_pred_prob_lstm = lstm_model.predict(X_test_pad)
y_pred_lstm = np.argmax(y_pred_prob_lstm, axis=1)

target_names = [ID_TO_LABEL[i] for i in range(NUM_CLASSES)]

print("LSTM Classification report:\n")
print(
    classification_report(
        y_test,
        y_pred_lstm,
        target_names=target_names,
        zero_division=0
    )
)

print("LSTM Confusion matrix (rows = true, cols = pred):")
print(confusion_matrix(y_test, y_pred_lstm))

382/382 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
LSTM Classification report:

              precision    recall  f1-score   support

    Negative       0.93      0.90      0.91      4463
     Neutral       0.92      0.89      0.90      3611
    Positive       0.87      0.92      0.90      4122

    accuracy                           0.90     12196
   macro avg       0.91      0.90      0.90     12196
weighted avg       0.91      0.90      0.90     12196

LSTM Confusion matrix (rows = true, cols = pred):
[[4026  148  289]
 [ 148 3202  261]
 [ 177  138 3807]]


An LSTM-based model was used for sentiment classification of tweets. It effectively captures word sequence information and achieves 90% accuracy, with balanced performance across negative, neutral, and positive classes using class weighting and early stopping to prevent overfitting.

### BiLSTM Model

In [74]:
# Build BiLSTM model

from tensorflow.keras.layers import Bidirectional, LSTM

model = models.Sequential([

    layers.Embedding(
        input_dim=MAX_NUM_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH
    ),

    layers.SpatialDropout1D(0.1),

    Bidirectional(LSTM(128)),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(NUM_CLASSES, activation="softmax")
])


model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [75]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

#### Train the model with EarlyStopping and class weights

In [76]:
# ---------------------------------------------------------
# Compute class weights and train model
# ---------------------------------------------------------

# Ensure numeric arrays
X_train_pad = np.asarray(X_train_pad, dtype="int32")
X_test_pad  = np.asarray(X_test_pad, dtype="int32")
y_train     = np.asarray(y_train, dtype="int32")
y_test      = np.asarray(y_test, dtype="int32")

In [77]:
# Compute class weights to handle any imbalance / ignored classes
classes = np.array([0, 1, 2])  # negative, neutral, positive

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

In [78]:
class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_array)}
print("Class weights:", class_weight_dict)

Class weights: {0: 0.9109293596997368, 1: 1.1259520841988644, 2: 0.9861127956337175}


In [79]:
# EarlyStopping callback
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# Train model
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_test_pad, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1,
    class_weight=class_weight_dict
)

Epoch 1/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - accuracy: 0.6287 - loss: 0.8041 - val_accuracy: 0.8388 - val_loss: 0.4168
Epoch 2/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.8933 - loss: 0.2934 - val_accuracy: 0.8839 - val_loss: 0.2985
Epoch 3/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.9361 - loss: 0.1740 - val_accuracy: 0.8941 - val_loss: 0.2961
Epoch 4/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - accuracy: 0.9530 - loss: 0.1273 - val_accuracy: 0.9070 - val_loss: 0.3007
Epoch 5/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - accuracy: 0.9638 - loss: 0.0971 - val_accuracy: 0.9133 - val_loss: 0.2929
Epoch 6/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.9704 - loss: 0.0774 - val_accuracy: 0.9157 - val_loss: 0.2989
Epoch 7/100
1525/1525 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.9746 - loss: 0.0646 - val_accuracy: 0.9131 - val_loss: 0.3175


#### Evaluate the model on the test set

In [80]:
# ---------------------------------------------------------
# Evaluate on test set
# ---------------------------------------------------------

y_pred_prob = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_prob, axis=1)

target_names = [ID_TO_LABEL[i] for i in range(NUM_CLASSES)]

print("Classification report:\n")
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

print("Confusion matrix (rows = true, cols = pred):")
print(confusion_matrix(y_test, y_pred))

382/382 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Classification report:

              precision    recall  f1-score   support

    Negative       0.93      0.91      0.92      4463
     Neutral       0.91      0.91      0.91      3611
    Positive       0.90      0.92      0.91      4122

    accuracy                           0.91     12196
   macro avg       0.91      0.91      0.91     12196
weighted avg       0.91      0.91      0.91     12196

Confusion matrix (rows = true, cols = pred):
[[4048  176  239]
 [ 130 3289  192]
 [ 168  153 3801]]


A Bidirectional LSTM (BiLSTM) model was used to improve sentiment classification by learning contextual information from both past and future words in a tweet. The embedding layer converts text into meaningful numerical vectors, while the BiLSTM captures richer semantic relationships compared to a standard LSTM. Class weights are applied to handle data imbalance, and early stopping prevents overfitting. Overall, the BiLSTM model provides more accurate and stable sentiment predictions across negative, neutral, and positive classes.

### BERT model

In [81]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch

##### Load BERT tokenizer

In [ ]:
BERT_MODEL_NAME = "bert-base-uncased"

tokenizer_bert = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

In [83]:
# Tokenizing Tweets
MAX_LEN = 64

def bert_tokenize(texts):
    return tokenizer_bert(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

X_train_enc = bert_tokenize(X_train)
X_test_enc  = bert_tokenize(X_test)

In [84]:
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

In [85]:
import torch

class TwitterDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = TwitterDataset(X_train_enc, y_train_tensor)
test_dataset  = TwitterDataset(X_test_enc, y_test_tensor)

In [ ]:
model_bert = BertForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=NUM_CLASSES
)

#### Train with BERT

In [87]:
training_args = TrainingArguments(
    output_dir="./bert_results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="no"
)

In [88]:
trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Step,Training Loss
100,0.958193
200,0.832016
300,0.766440
400,0.719843
500,0.746041
600,0.705334
700,0.717709
800,0.703484
900,0.665331
1000,0.621910


TrainOutput(global_step=9147, training_loss=0.31819686199441377, metrics={'train_runtime': 1669.3266, 'train_samples_per_second': 87.669, 'train_steps_per_second': 5.479, 'total_flos': 4813298196384384.0, 'train_loss': 0.31819686199441377, 'epoch': 3.0})

#### Evaluate the model on test set

In [89]:
preds = trainer.predict(test_dataset)
y_pred_bert = preds.predictions.argmax(axis=1)

print("BERT Classification Report:\n")
print(classification_report(
    y_test,
    y_pred_bert,
    target_names=[ID_TO_LABEL[i] for i in range(NUM_CLASSES)],
    zero_division=0
))

print("BERT Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_bert))

BERT Classification Report:

              precision    recall  f1-score   support

    Negative       0.94      0.94      0.94      4463
     Neutral       0.93      0.91      0.92      3611
    Positive       0.92      0.93      0.93      4122

    accuracy                           0.93     12196
   macro avg       0.93      0.93      0.93     12196
weighted avg       0.93      0.93      0.93     12196

BERT Confusion Matrix:
[[4187  118  158]
 [ 139 3300  172]
 [ 150  133 3839]]


A BERT-based transformer model was fine-tuned for Twitter sentiment analysis. The BERT tokenizer captures sub-word information and full sentence context, allowing the model to understand informal and short text more effectively. With minimal fine-tuning, the model delivers strong and consistent performance across negative, neutral, and positive sentiment classes.

Overall, the BERT model outperforms traditional LSTM and BiLSTM models by providing more accurate and context-aware sentiment predictions, making it the most effective model which is also the final model.

### Final Model

In [ ]:
SAVE_DIR = "./bert_sentiment_model"

In [ ]:
trainer.save_model(SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
tokenizer_bert.save_pretrained(SAVE_DIR)

('./bert_sentiment_model/tokenizer_config.json',
 './bert_sentiment_model/tokenizer.json')

In [ ]:
import os
os.listdir(SAVE_DIR)

['config.json',
 'model.safetensors',
 'tokenizer_config.json',
 'training_args.bin',
 'tokenizer.json']

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("bert_sentiment_model", "zip", SAVE_DIR)
files.download("bert_sentiment_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# used google colab with T4 GPU hardware accelerator to run BERT model.